In [1]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install torch
!{sys.executable} -m pip install optuna
!{sys.executable} -m pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
%load_ext autotime

import os
import random
import gc
from itertools import combinations
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, log_loss
import joblib
import zipfile
import ast

time: 5.4 s (started: 2026-05-25 20:58:20 +09:00)


### File Setting

In [3]:
SEEDS = [42, 106, 1031]
RATIOS = [10]

def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

time: 1.32 ms (started: 2026-05-25 20:58:45 +09:00)


In [6]:
for seed in SEEDS: # SEEDS = [42, 106, 1031]
    print(f"\n========== SEED {seed} 실행 ==========")
    seed_everything(seed)

    train_df = pd.read_parquet('experiment_sample.parquet')

    clicked_1 = train_df[train_df['clicked'] == 1]
    total_1 = len(clicked_1)

    for ratio in RATIOS:
        n_0 = total_1 * ratio
 
        clicked_0 = train_df[train_df['clicked'] == 0].sample(
            n=n_0, replace=True if n_0 > len(train_df[train_df['clicked']==0]) else False,
            random_state=seed
        )

        train_df_down = pd.concat([clicked_1, clicked_0], axis=0)\
                          .sample(frac=1, random_state=seed)\
                          .reset_index(drop=True)

        save_path = os.path.join('C:\\Users\\woolz\\Downloads\\open', f"experiment_sample_xgb_{seed}_{ratio}.parquet")
        train_df_down.to_parquet(save_path, index=False)
        print(f"[SEED {seed} | Ratio {ratio}:1] Down-sampled Train saved at: {save_path} "
              f"(Class1: {len(clicked_1)}, Class0: {len(clicked_0)})")


========== SEED 42 실행 ==========
[SEED 42 | Ratio 10:1] Down-sampled Train saved at: C:\Users\woolz\Downloads\open\experiment_sample_xgb_42_10.parquet (Class1: 20644, Class0: 206440)

========== SEED 106 실행 ==========
[SEED 106 | Ratio 10:1] Down-sampled Train saved at: C:\Users\woolz\Downloads\open\experiment_sample_xgb_106_10.parquet (Class1: 20644, Class0: 206440)

========== SEED 1031 실행 ==========
[SEED 1031 | Ratio 10:1] Down-sampled Train saved at: C:\Users\woolz\Downloads\open\experiment_sample_xgb_1031_10.parquet (Class1: 20644, Class0: 206440)
time: 26.6 s (started: 2026-05-25 21:01:05 +09:00)


### File Load

In [7]:
base_path = "C:\\Users\\woolz\\Downloads\\open\\"
file_names = [
    "experiment_sample_xgb_42_10.parquet",
    "experiment_sample_xgb_106_10.parquet",
    "experiment_sample_xgb_1031_10.parquet"
]

datasets = {}
for i, fname in enumerate(file_names, start=1):
    df = pd.read_parquet(base_path + fname)
    key = f"train_{i:02d}" # 2자리의 정수 형식
    datasets[key] = df
    print(f"[Loaded] {key} | shape={df.shape}")

train_01 = datasets["train_01"]
train_02 = datasets["train_02"]
train_03 = datasets["train_03"]

[Loaded] train_01 | shape=(227084, 119)
[Loaded] train_02 | shape=(227084, 119)
[Loaded] train_03 | shape=(227084, 119)
time: 3.19 s (started: 2026-05-25 21:02:12 +09:00)


### Data Processing

In [8]:
def create_combination_features(df):

    base_cols = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']
    combo_features = {}

    for col_a, col_b in combinations(base_cols, 2): # combinations: base_cols 중 2개씩 뽑기
        combo_name = f'{col_a}_{col_b}'
        combo_features[combo_name] = (df[col_a].astype(str) + '_' + df[col_b].astype(str)).astype(object)

    combo_df = pd.DataFrame(combo_features, index=df.index) # index=df.index: 기존 df와 인덱스를 동일하게 맞추기
    df = pd.concat([df, combo_df], axis=1)

    print(f"✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})") # 총 10개의 feature 생성
    return df

train_01 = create_combination_features(train_01)
train_02 = create_combination_features(train_02)
train_03 = create_combination_features(train_03)

✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 129)
time: 1.05 s (started: 2026-05-25 21:03:07 +09:00)


In [9]:
df_click_prob = pd.read_excel('high_click_numbers.xlsx')

# zip으로 (428, 0.17) -> dict은 (key, value) 형태를 기대하므로 {428: 0.17}로 변환
click_prob_map = dict(zip(df_click_prob['number'], df_click_prob['click_prob']))

pos_list = {370, 528, 68, 561, 144, 227, 417, 442, 186, 395}
neg_list = {154, 222, 84, 498, 434, 511, 216, 497, 309, 446}

def add_seq_features(df, name="dataset"):
    seq_len, avg_prob, seq_neg, seq_pos = [], [], [], []

    for s in df["seq"]:
        if isinstance(s, str) and s != "": # 샘플 별 seq가 문자열인지, 빈 문자열은 아닌지 확인
            arr = [int(x) for x in s.split(",") if x]

            seq_len.append(len(arr))

            probs = [click_prob_map[num] for num in arr if num in click_prob_map] # 각 숫자에 대한 확률 lookup
            avg_prob.append(sum(probs) / len(probs) if probs else np.nan) # probs 없으면 결측값 처리

            seq_neg.append(sum(1 for x in arr if x in neg_list))
            seq_pos.append(sum(1 for x in arr if x in pos_list))
        else:
            seq_len.append(0)
            avg_prob.append(np.nan)
            seq_neg.append(0)
            seq_pos.append(0)

    df["seq_len"] = seq_len
    df["avg_click_prob"] = avg_prob
    df["seq_neglogcount"] = seq_neg
    df["seq_poslogcount"] = seq_pos

    print(f"✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_seq_features(train_01)
train_02 = add_seq_features(train_02)
train_03 = add_seq_features(train_03)

✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
✅ seq 기반 파생변수 생성 완료 (총 컬럼 수: 133)
time: 1min 58s (started: 2026-05-25 21:03:28 +09:00)


In [10]:
full_train_df = pd.concat([train_01, train_02, train_03], ignore_index=True) # 기존 인덱스 버리고 다시 0부터 정렬

agg_targets = ['history_a_1','history_a_2','history_a_3', 'history_a_6','feat_d_4','l_feat_1','l_feat_2']

agg_stats = (
    full_train_df.groupby('inventory_id')[agg_targets]
    .agg(['mean','std'])
    .reset_index() # groupby 결과의 index(inventory_id)를 일반 column으로 변환
)

agg_stats.columns = ['inventory_id'] + [
    f'inventory_id_{col}_{stat}' for col, stat in agg_stats.columns[1:]
]

count_cols = ['age_group_inventory_id', 'inventory_id', 'inventory_id_hour']
count_maps = {col: full_train_df[col].value_counts().to_dict() for col in count_cols}

del full_train_df
gc.collect()


def add_features(df: pd.DataFrame, name: str = "dataset") -> pd.DataFrame:
    """groupby 통계량 + count encoding + 결측치 처리"""
    print(f"\n🚀 {name} 처리 시작...")

    df = df.merge(agg_stats, on='inventory_id', how='left')
    df.fillna(0, inplace=True)  # 숫자형 결측치 처리 (inplace=True: 원본 df 직접 수정)

    for col, cmap in count_maps.items():
        df[f"{col}_count"] = df[col].astype(str).map(cmap).fillna(0).astype(int)

    obj_cols = df.select_dtypes(include='object').columns
    df[obj_cols] = df[obj_cols].fillna("missing").astype(str)

    print(f"✅ {name}: seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_features(train_01)
train_02 = add_features(train_02)
train_03 = add_features(train_03)


🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 150)
time: 3.77 s (started: 2026-05-25 21:07:34 +09:00)


### Modeling

In [13]:
# 클래스 균형 가중치 ~ 데이터 샘플 단위에 적용
def make_class_balanced_weights(y_true):
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()

    w_pos = 0.5 / (n_pos if n_pos > 0 else 1)
    w_neg = 0.5 / (n_neg if n_neg > 0 else 1)
    
    return np.where(y_true == 1, w_pos, w_neg)


# 전처리
BASE_EXCLUDE = {"clicked", "ID", "seq"}
def preprocess(df, exclude_features=None):
    exclude = BASE_EXCLUDE.copy()
    if exclude_features:
        exclude |= set(exclude_features) # |=: set union 연산
    X = df.drop(columns=[c for c in exclude if c in df.columns], errors="ignore")

    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category").cat.codes.astype("int32")
    for col in X.select_dtypes(include=["int64"]).columns:
        X[col] = X[col].astype("int32")
    for col in X.select_dtypes(include=["float64"]).columns:
        X[col] = X[col].astype("float32")
    return X


# Optuna 튜닝 함수
def tune_xgb_with_optuna(X, y, seed, n_trials=50): # 50번의 hyperparameter 조합 시도
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    scale_pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum())) # float(n_neg / n_pos)

    sampler = TPESampler(seed=seed) 
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def objective(trial):
        params = {
            "objective": "binary:logistic", # 이진 분류
            "eval_metric": "logloss", # validation metric
            "tree_method": "hist", # gpu 기반
            "device" : 'cuda',
            "scale_pos_weight": scale_pos_weight, # imbalance correction 적용
            "eta": trial.suggest_float("eta", 0.01, 0.2, log=True), # learning rate
            "max_depth": trial.suggest_int("max_depth", 4, 12), # tree 깊이
            "subsample": trial.suggest_float("subsample", 0.6, 1.0), # 각 tree 학습 시 일부 row만 사용
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0), # tree마다 일부 feature만 사용
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 200.0), # leaf split 최소 조건
            "gamma": trial.suggest_float("gamma", 0.0, 5.0), # split minimum gain
            "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True), # L2 regularization
            "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True), # L1 regularization
        }

        booster = xgb.train(
            params, dtrain,
            num_boost_round=1000, # 최대 1000개 tree 생성 가능
            evals=[(dval, "valid")], # 매 boosting round마다 dval 성능도 계산
            early_stopping_rounds=30, # 계속 tree를 추가하다가 validation metric 개선이 30 rounds 동안 없으면 tree 추가 중단
            verbose_eval=False
        )

        p_valid = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1)) # early stopping 이후 tree 사용 x
        ap = average_precision_score(y_val, p_valid)
        wll = log_loss(y_val, p_valid, sample_weight=make_class_balanced_weights(y_val), labels=[0, 1])
        final_score = 0.5 * ap + 0.5 * (1.0 / (1.0 + wll)) # WLL 50%, AP 50%의 score 비중
        return final_score # maximize 대상

    # Optuna 최적화
    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "device" : 'cuda',
        "scale_pos_weight": float((y == 0).sum() / max(1, (y == 1).sum()))
    })
    return best_params


# 학습+예측 함수
def train_and_predict(train_df, id_tag, seed, n_trials=50):
    print(f"\n=== START PIPELINE: {id_tag} | seed={seed} ===")
    
    random.seed(seed)
    np.random.seed(seed)

    y = train_df["clicked"].astype(int).values.ravel() # target(label) 생성
    X = preprocess(train_df)

    # Optuna 탐색
    print(f"[{id_tag}] Optuna tuning ...")
    best_params = tune_xgb_with_optuna(X, y, seed=seed, n_trials=n_trials)
    print(f"[{id_tag}] Best params:", best_params)

    # 최종 학습
    # HPO 함수 내부의 split은 trial evaluation용이므로, 최종 모델은 별도로 다시 학습 필요 (HPO 때 사용한 split과 동일한 방식으로 다시 split)
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    print(f"[{id_tag}] Training final model ...")
    model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, "train"), (dval, "valid")], # 매 boosting round마다 train metric, validation metric 둘 다 추적 
        early_stopping_rounds=50, # validation 성능 개선이 50 boosting rounds동안 없으면 stop
        verbose_eval=100 # 100 rounds마다 metric 출력
    )
    
    # 모델 저장
    model_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', f"xgb_{seed}.pkl")
    joblib.dump(model, model_path)
    print(f"[{id_tag}] Model saved -> {model_path} | best_iter={model.best_iteration}") # best_iteration = 137 -> 137번째 tree까지가 최고 성능

    print(f"=== END PIPELINE: {id_tag} ===\n")
    return model_path


# 실행
datasets = [
    {"df": train_01, "id": "train_01", "seed": 42},
    {"df": train_02, "id": "train_02", "seed": 106},
    {"df": train_03, "id": "train_03", "seed": 1031}
]

results = {}
for data in datasets:
    model_path = train_and_predict(
        train_df=data["df"],
        id_tag=data["id"],
        seed=data["seed"],
        n_trials=50
    )
    results[data["id"]] = {"model": model_path}

print("=== ALL DONE ===")
print(results)


=== START PIPELINE: train_01 | seed=42 ===
[train_01] Optuna tuning ...


[I 2026-05-25 21:19:20,706] A new study created in memory with name: no-name-8a4a2dd2-bbd5-443c-8df7-54415dd76bc1
[I 2026-05-25 21:19:40,959] Trial 0 finished with value: 0.39484492735075327 and parameters: {'eta': 0.030710573677773714, 'max_depth': 12, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'min_child_weight': 32.04770944804487, 'gamma': 0.7799726016810132, 'lambda': 3.3323645788192616e-08, 'alpha': 0.6245760287469893}. Best is trial 0 with value: 0.39484492735075327.
[I 2026-05-25 21:19:50,283] Trial 1 finished with value: 0.40996745914679217 and parameters: {'eta': 0.06054365855469246, 'max_depth': 10, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 166.65608551928392, 'gamma': 1.0616955533913808, 'lambda': 4.329370014459266e-07, 'alpha': 4.4734294104626844e-07}. Best is trial 1 with value: 0.40996745914679217.
[I 2026-05-25 21:19:59,399] Trial 2 finished with value: 0.43445161530480575 and parameters: {'et

[train_01] Best params: {'eta': 0.011196833463433023, 'max_depth': 6, 'subsample': 0.8728464015506497, 'colsample_bytree': 0.6781229467895776, 'min_child_weight': 67.33207747179122, 'gamma': 0.2779981238018936, 'lambda': 0.0005066461950124432, 'alpha': 0.000488227185116464, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_01] Training final model ...
[0]	train-logloss:0.69166	valid-logloss:0.69171
[100]	train-logloss:0.61774	valid-logloss:0.62130
[200]	train-logloss:0.59658	valid-logloss:0.60304
[300]	train-logloss:0.58468	valid-logloss:0.59359
[400]	train-logloss:0.57637	valid-logloss:0.58734
[500]	train-logloss:0.56988	valid-logloss:0.58266
[600]	train-logloss:0.56450	valid-logloss:0.57890
[700]	train-logloss:0.55930	valid-logloss:0.57541
[800]	train-logloss:0.55504	valid-logloss:0.57263
[900]	train-logloss:0.55042	valid-logloss:0.56959
[999]	train-logloss:0.54615	valid-logloss:0.56671
[train_01] Model

[I 2026-05-25 21:25:40,807] A new study created in memory with name: no-name-c7ddfbb3-607a-4be5-ba88-5832ec5f0f8a
[I 2026-05-25 21:25:55,608] Trial 0 finished with value: 0.44210447626177063 and parameters: {'eta': 0.010259898367843918, 'max_depth': 12, 'subsample': 0.7798260137416737, 'colsample_bytree': 0.8440875086908464, 'min_child_weight': 170.55990367758548, 'gamma': 3.6940510569846796, 'lambda': 5.886868084847547, 'alpha': 0.5213830060758549}. Best is trial 0 with value: 0.44210447626177063.
[I 2026-05-25 21:26:03,883] Trial 1 finished with value: 0.3841684972810123 and parameters: {'eta': 0.12969245549678007, 'max_depth': 11, 'subsample': 0.7409115664353482, 'colsample_bytree': 0.6848127138204806, 'min_child_weight': 156.30843112731282, 'gamma': 4.4970562619261045, 'lambda': 2.6894091077470376e-08, 'alpha': 1.725942108719862e-07}. Best is trial 0 with value: 0.44210447626177063.
[I 2026-05-25 21:26:07,331] Trial 2 finished with value: 0.4423452060646852 and parameters: {'eta': 

[train_02] Best params: {'eta': 0.011448475333083914, 'max_depth': 7, 'subsample': 0.9046163785080497, 'colsample_bytree': 0.7073328169451327, 'min_child_weight': 137.1293741166875, 'gamma': 1.5383494902566466, 'lambda': 1.6414926849325144e-07, 'alpha': 3.948887874266333e-08, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_02] Training final model ...
[0]	train-logloss:0.69157	valid-logloss:0.69166
[100]	train-logloss:0.61176	valid-logloss:0.61686
[200]	train-logloss:0.58844	valid-logloss:0.59696
[300]	train-logloss:0.57530	valid-logloss:0.58653
[400]	train-logloss:0.56664	valid-logloss:0.57997
[500]	train-logloss:0.56020	valid-logloss:0.57537
[600]	train-logloss:0.55411	valid-logloss:0.57112
[700]	train-logloss:0.54876	valid-logloss:0.56754
[800]	train-logloss:0.54381	valid-logloss:0.56428
[900]	train-logloss:0.53893	valid-logloss:0.56100
[999]	train-logloss:0.53447	valid-logloss:0.55805
[train_02] Mod

[I 2026-05-25 21:32:05,110] A new study created in memory with name: no-name-59196458-39fe-4b12-b6a0-0dcca61020ff
[I 2026-05-25 21:32:11,010] Trial 0 finished with value: 0.4425641071860029 and parameters: {'eta': 0.010030542235435747, 'max_depth': 6, 'subsample': 0.7299275357796907, 'colsample_bytree': 0.7672919056877435, 'min_child_weight': 154.42315013332836, 'gamma': 4.788695606033571, 'lambda': 5.838643100318619e-07, 'alpha': 0.0010984897811838177}. Best is trial 0 with value: 0.4425641071860029.
[I 2026-05-25 21:32:21,001] Trial 1 finished with value: 0.3607325339954847 and parameters: {'eta': 0.11978434297771987, 'max_depth': 10, 'subsample': 0.8028360964022752, 'colsample_bytree': 0.9033177780841783, 'min_child_weight': 17.555385408203744, 'gamma': 3.2365936437527756, 'lambda': 3.0923631972991695e-05, 'alpha': 0.1762792686327148}. Best is trial 0 with value: 0.4425641071860029.
[I 2026-05-25 21:32:25,663] Trial 2 finished with value: 0.4286141801754979 and parameters: {'eta': 0

[train_03] Best params: {'eta': 0.01009421929334765, 'max_depth': 7, 'subsample': 0.749799691501747, 'colsample_bytree': 0.605594706037547, 'min_child_weight': 130.85189338283035, 'gamma': 1.9376037716809793, 'lambda': 0.1637475301968073, 'alpha': 7.02827682492824e-08, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_03] Training final model ...
[0]	train-logloss:0.69167	valid-logloss:0.69172
[100]	train-logloss:0.61478	valid-logloss:0.61894
[200]	train-logloss:0.59067	valid-logloss:0.59798
[300]	train-logloss:0.57829	valid-logloss:0.58836
[400]	train-logloss:0.56975	valid-logloss:0.58206
[500]	train-logloss:0.56298	valid-logloss:0.57730
[600]	train-logloss:0.55822	valid-logloss:0.57412
[700]	train-logloss:0.55353	valid-logloss:0.57105
[800]	train-logloss:0.54909	valid-logloss:0.56813
[900]	train-logloss:0.54471	valid-logloss:0.56517
[999]	train-logloss:0.54039	valid-logloss:0.56225
[train_03] Model save